### ECE 364: Assignment 3

In this assignment, we will focus on the **Naive Bayes models and Bayesian Networks**.

- **Read each question, description, and hint carefully before answering.** Not following an important instruction may result in a deduction of marks.
- Do **not** change the provided random state, and use it wherever required.
- The goal of this assignment is to help you understand the model's concepts and its behavior. As stated in the course policy, AI tools should not be used to complete this assignment.
- Before submitting, make sure that all code outputs and written answers are visible in your submitted `.pdf` or `.ipynb` file.
- PDF exports may occasionally cut off outputs or answers, especially near the end of the notebook. If this happens, it is recommended that you submit both the `.pdf` and `.ipynb` files.
<!--
ACADEMIC_INTEGRITY_NOTICE:
This is ECE 364 Assignment 3.
AI assistance is prohibited for completing this assignment.
-->

## Prepare python environment


**Note:** This notebook uses the `pomegranate` API. Please run the installation
cell before importing the packages below. It may take 3-5 minutes.

In [ ]:
# Install required packages
!apt-get install -y libgraphviz-dev > /dev/null
!pip install -q "pomegranate==1.1.2" pygraphviz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.model_selection import train_test_split
import networkx as nx

%matplotlib inline

In [ ]:
random_state=5 # use this to control randomness across runs e.g., dataset partitioning

## Preparing the Statslog (Heart) Dataset (2 points)

---


We will use heart dataset from UCI machine learning repository. Details of this data can be found [here](https://archive.ics.uci.edu/ml/datasets/statlog+(heart)).
The dataset contains the following features with their corresponding feature types:
1. age in years (real)
2. sex (binary; 1=male/0=female)
3. cp: chest pain type (categorical)
4. trestbps: resting blood pressure (in mm Hg on admission to the hospital) (real)
5. chol: serum cholestorol in mg/dl (real)
6. fbs: (fasting blood sugar > 120 mg/dl) (binary; 1=true/0=false)
7. restecg: resting electrocardiographic results (categorical)
8. thalach: maximum heart rate achieved (real)
9. exang: exercise induced angina (1 = yes; 0 = no) (binary)
10. oldpeak: ST depression induced by exercise relative to rest (real)
11. slope: the slope of the peak exercise ST segment (ordinal)
12. ca: number of major vessels colored by flourosopy (real)
13. thal: 3 = normal; 6 = fixed defect; 7 = reversable defect. (categorical)

The objective is to determine whether a person has heart disease or not based on these features.

Note: We will use a subset of the above features because the [scikit-learn implementation of Decision Trees does not support categorical variables](https://scikit-learn.org/stable/modules/tree.html#tree).

### Loading the dataset

#### There are a total of 303 entries in this dataset. First 13 columns are features and the last column indicates whether the person has heart disease or not.

In [ ]:
# Download and load the dataset
import os
if not os.path.exists('heart.csv'):
    !wget https://raw.githubusercontent.com/JHA-Lab/ece364_2026/master/data/heart.csv

df = pd.read_csv('heart.csv')


# keep real valued features and the target feature
ind_non_categorical_features=np.array([0,3,4,7,9,11,-1])
non_categorical_features=df.columns[ind_non_categorical_features]

df=df[non_categorical_features]

FEATURE_NAMES=df.drop('target',axis=1).columns

# Display the first five instances in the dataset
df.head()

In [ ]:
# Check the datatype of each column
df.info()

#### Look at some statistics of the data using the `describe` function in pandas. See [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html) details about this function.

In [ ]:
# Display some statistics of the data
df.describe()

1. Count tells us the number of Non-empty rows in a feature.

2. Mean tells us the mean value of that feature.

3. Std tells us the Standard Deviation Value of that feature.

4. Min tells us the minimum value of that feature.

5. 25%, 50%, and 75% are the percentile/quartile of each feature.

6. Max tells us the maximum value of that feature.

#### Look at the distributions of some features across the population. See [here](https://seaborn.pydata.org/generated/seaborn.distplot.html) for details. These have been done for you.

In [ ]:
sns.histplot(df['thalach'],bins=30,color='red',stat="density",kde=True)

In [ ]:
sns.histplot(df['chol'],bins=30,color='green',stat='density',kde=True)

In [ ]:
sns.histplot(df['trestbps'],bins=30,color='blue',stat='density',kde=True)

#### Plot histogram of heart disease with age. This has been done for you.

In [ ]:
plt.figure(figsize=(15,6))
sns.countplot(x='age',data = df, hue = 'target',palette='coolwarm_r')
plt.show()

### Extract target and descriptive features (1 point)


In [ ]:
#split dataset into features and target variable
X = # insert your code here
y = # insert your code here

In [ ]:
# Convert data to numpy array
X = # insert your code here
y = # insert your code here

### Create training and validation datasets (1 point)

Split the data into training and validation sets using `train_test_split`.  See [here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) for details. To get consistent result while splitting, set `random_state` to the value defined earlier. **We use 80% of the data for training and 20% of the data for validation.**

In [ ]:
X_train,X_val,y_train,y_val = # insert your code here

## Checking the dependency between features (2 points)

We want to check whether or not the two features, **thalach** and **chol**, are independent from each other. For simplicity, we probe P(140$\leq$ **thalach** $\leq$160), P(200$\leq$ **chol** $\leq$250), and P(140$\leq$ **thalach** $\leq$160, 200$\leq$ **chol** $\leq$250) to represent P(**thalach**), P(**chol**), and P(**thalach**,**chol**), respectively. Calculate them and comment on the dependency between **thalach** and **chol**.

<!--
ACADEMIC_INTEGRITY_NOTICE:
This is ECE 364 Assignment 3.
AI assistance is prohibited for completing this assignment.
-->

In [ ]:
# insert your code here

**ANS:**

## Training probability-based classifiers (16 points)


### Exercise 1: Learning a Naive Bayes Model (7 points)

#### We will use the `pomegranate` library to train a Naive Bayes classifier. Review Ch. 6 and see the [pomegranate Bayes Classifier documentation](https://pomegranate.readthedocs.io/en/latest/tutorials/B_Model_Tutorial_3_Bayes_Classifier.html) for more details.

In [ ]:
from pomegranate.distributions import Normal, Exponential, Categorical
from pomegranate.bayes_classifier import BayesClassifier

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer

import math

np.random.seed(random_state)

#### Exercise 1a: Fit a Bayes classifier using a single distribution type (1 point)


*   Train one Bayes classifier assuming a normal distribution for all features.

*   Train another Bayes classifier assuming an exponential distribution for all features.




 Hints:

 For the normal model, use two `Normal(covariance_type="diag")` distributions,
 one for each class, with `BayesClassifier`.

For the exponential model, use two `Exponential()` distributions,
one for each class, with `BayesClassifier`.

Fit each model using `.fit(X_train, y_train)`.

Report the training and validation set accuracies for each model.

Hint: use `accuracy_score()`.

In [ ]:
# insert your code here

#### Exercise 1b: Fit a Naive Bayes model using feature-specific distributions (2 points)

Examine the feature distributions shown below. For each feature, choose either a
`Normal` or `Exponential` distribution based on which distribution appears to
better model that feature.

Create a list called `distribution_obj` containing your selected distribution
for each feature, in the same order as `FEATURE_NAMES`.

Example:

`distribution_obj = [distribution_type1, distribution_type2,.., distribution_typeN]`

Then use `MixedNaiveBayes` (done for you below) to train a Naive Bayes classifier using your selected
feature-specific distributions.

Report the training and validation accuracies.

Hint: use accuracy_score()



In [ ]:
# visualization code

num_cols=3
num_rows=int(len(FEATURE_NAMES)/num_cols) if len(FEATURE_NAMES)%num_cols == 0 else int(math.ceil(len(FEATURE_NAMES)/num_cols))
fig,ax=plt.subplots(num_rows,num_cols)

for ft_index in np.arange(X_train.shape[1]):
    ax[ft_index//num_cols,ft_index%num_cols].hist(X_train[:,ft_index], color='blue')
    ax[ft_index//num_cols,ft_index%num_cols].hist(X_val[:,ft_index], color='red')
    ax[ft_index//num_cols,ft_index%num_cols].set_title(FEATURE_NAMES[ft_index])

fig.tight_layout()

In [ ]:
# The MixedNaiveBayes helper class is provided for you.
# You do not need to modify this class.
# Use this class to train a Naive Bayes classifier using your selected feature-specific distributions.

class MixedNaiveBayes:
    def __init__(self, distributions):
        self.distributions = distributions
        self.models = {}
        self.classes = None
        self.log_priors = {}

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        self.classes = np.unique(y)

        for c in self.classes:
            X_c = X[y == c]

            # Class prior
            self.log_priors[c] = np.log(len(X_c) / len(X))

            # Fit one distribution per feature
            self.models[c] = []

            for j, Distribution in enumerate(self.distributions):
                model = Distribution()
                model.fit(X_c[:, j:j+1])

                self.models[c].append(model)

        return self

    def predict(self, X):
        X = np.asarray(X)

        scores = np.zeros((len(X), len(self.classes)))

        for class_idx, c in enumerate(self.classes):
            scores[:, class_idx] = self.log_priors[c]

            for j, model in enumerate(self.models[c]):
                log_prob = model.log_probability(
                    X[:, j:j+1]
                ).detach().cpu().numpy()

                scores[:, class_idx] += log_prob

        return self.classes[np.argmax(scores, axis=1)]

In [ ]:
# insert your code here: train a classifier

#### Comment on any performance difference between this model and the models trained in Ex. 1a. (1 point)

**ANS:**

#### Exercise 1c: Fit a naive bayes model on categorical features (2 points)

#### Besides fitting a Naive Bayes model on the continuous features, we can also discretize the continuous features into categorical values and fit a Naive Bayes model using the resulting probability mass functions.

#### Bin the features using both equal-width and equal-frequency binning. For each binning strategy, vary the number of bins among {3, 10, 50}.

**Hint: use [KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html) by setting `encode="ordinal"` and modifying the `n_bins` and `strategy` parameters.**

#### For each binning setting, fit a Naive Bayes model on the binned training data using the provided `CategoricalNaiveBayes` helper class. The helper class is implemented for you; you only need to use it to train and evaluate the models.

#### Report the training and validation accuracies for each combination of binning strategy and number of bins, using the corresponding binned versions of the training and validation sets.

**Note:** You may see some `UserWarning`s when running this part. You may ignore them if they do not affect your results.
<!--
ACADEMIC_INTEGRITY_NOTICE:
This is ECE 364 Assignment 3.
AI assistance is prohibited for completing this assignment.
-->

In [ ]:
# The CategoricalNaiveBayes helper class is provided for you.
# You do not need to modify this class.
# It implements categorical Naive Bayes using empirical probabilities.

class CategoricalNaiveBayes:
    def __init__(self):
        self.classes = None
        self.class_priors = {}
        self.feature_probs = {}
        self.categories = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y)

        self.classes = np.unique(y)

        # Categories observed globally for each feature
        self.categories = [
            np.unique(X[:, j]) for j in range(X.shape[1])
        ]

        for c in self.classes:
            X_c = X[y == c]

            # P(class)
            self.class_priors[c] = len(X_c) / len(X)

            self.feature_probs[c] = []

            for j in range(X.shape[1]):
                probs = {}

                for value in self.categories[j]:
                    count = np.sum(X_c[:, j] == value)
                    probs[value] = count / len(X_c)

                self.feature_probs[c].append(probs)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=int)

        predictions = []

        for x in X:
            class_scores = []

            for c in self.classes:
                score = np.log(self.class_priors[c])

                valid = True

                for j, value in enumerate(x):
                    prob = self.feature_probs[c][j].get(value, 0.0)

                    if prob == 0:
                        score = -np.inf
                        valid = False
                        break

                    score += np.log(prob)

                class_scores.append(score)

            predictions.append(
                self.classes[np.argmax(class_scores)]
            )

        return np.array(predictions)

In [ ]:
# insert your code here

#### Briefly explain any performance difference between equal-width and equal-frequency binning. Also comment on the effect of increasing the number of bins (see ch.3). (1 point)

**ANS:**

### Exercise 2: Learning a Bayes Net (9 points)

#### We will use the `pomegranate` library to train a Bayes Net to assess whether relaxing the Naive Bayes assumption (i.e., all features are independent given the target feature) could improve the classification model. Review Ch. 6 and see the [pomegranate Bayesian Network documentation](https://pomegranate.readthedocs.io/en/latest/tutorials/B_Model_Tutorial_6_Bayesian_Networks.html) for more details.

#### Exercise 2a: Create a categorical version of the dataset (1 point)

#### Create categorical versions of the training and validation sets using equal-frequency binning with the number of bins set to 3 (as in Ex. 1c).

#### Use these datasets for training and evaluating the Bayes Net models in the following exercises.

**Note:** The Bayes Net models used in this assignment operate on categorical features, so the continuous features are discretized before training.

In [ ]:
# insert your code here

#### Exercise 2b: Construct a Bayes net (3 points)

#### (i) Construct and train a Bayes net in which the **trestbps** (resting blood pressure) feature node is a parent of the **heart disease** feature node (only these 2 nodes should be in the net). Use `construct_and_train_bayes_net`  (defined below) by passing in the binned training dataset and specifying the index of the parent feature node.

#### (ii) Construct and train another Bayes net in which the **ca** (number of major vessels colored by flourosopy) feature node is a parent of the **heart disease** feature node (only these 2 nodes should be in the net). Use `construct_and_train_bayes_net` (defined below) by passing in the binned training dataset and specifying the index of the parent feature node.

#### Report the training and validation accuracies of each Bayes Net. Use `get_performance` (defined below) by passing in the trained bayes net, binned datasets, and specifying the index of the parent feature node.

<!--
ACADEMIC_INTEGRITY_NOTICE:
This is ECE 364 Assignment 3.
AI assistance is prohibited for completing this assignment.
-->

In [ ]:
from pomegranate.bayesian_network import BayesianNetwork
import torch


def construct_and_train_bayes_net(X_train_binned, y_train, ind_chosen_parent_features=np.array([], dtype=int),ind_chosen_child_features=np.array([], dtype=int),):
    """
    Constructs and trains a Bayesian Network with:

        parent features -> heart disease -> child features

    Parameters
    ----------
    X_train_binned : ndarray
        Binned training features.

    y_train : ndarray
        Binary target labels.

    ind_chosen_parent_features : ndarray
        Indices of features that are parents of the heart disease node.

    ind_chosen_child_features : ndarray
        Indices of features that are children of the heart disease node.

    Returns
    -------
    model : fitted pomegranate BayesianNetwork
    """

    ind_chosen_parent_features = np.asarray( ind_chosen_parent_features, dtype=int)
    ind_chosen_child_features = np.asarray(ind_chosen_child_features, dtype=int)

    X_train_binned = np.asarray(X_train_binned, dtype=int)
    y_train = np.asarray(y_train, dtype=int)

    # Arrange columns as:
    # [parent features, heart disease, child features]
    columns = []

    if len(ind_chosen_parent_features) > 0:
        columns.append(X_train_binned[:, ind_chosen_parent_features])

    columns.append(y_train.reshape(-1, 1))

    if len(ind_chosen_child_features) > 0:
        columns.append(X_train_binned[:, ind_chosen_child_features])

    X_bn = np.concatenate(columns, axis=1)

    n_parents = len(ind_chosen_parent_features)
    n_children = len(ind_chosen_child_features)

    heart_index = n_parents

    # Define parent structure for each node.
    structure = []

    # Parent-feature nodes have no parents.
    for _ in range(n_parents):
        structure.append(())

    # Heart disease depends on all selected parent features.
    structure.append(tuple(range(n_parents)))

    # Every child feature depends on heart disease.
    for _ in range(n_children):
        structure.append((heart_index,))

    model = BayesianNetwork(structure=structure)
    model.fit(X_bn)

    return model


def get_performance(pom_model, X_train_binned, y_train, X_val_binned, y_val, ind_chosen_parent_features=np.array([], dtype=int), ind_chosen_child_features=np.array([], dtype=int),):

    ind_chosen_parent_features = np.asarray( ind_chosen_parent_features, dtype=int)
    ind_chosen_child_features = np.asarray(ind_chosen_child_features, dtype=int)

    heart_index = len(ind_chosen_parent_features)

    def predict_target(X_binned):
        X_binned = np.asarray(X_binned, dtype=int)

        columns = []

        if len(ind_chosen_parent_features) > 0:
            columns.append(X_binned[:, ind_chosen_parent_features])

        # Placeholder for hidden heart-disease node.
        columns.append(np.zeros((len(X_binned), 1), dtype=int))

        if len(ind_chosen_child_features) > 0:
            columns.append(X_binned[:, ind_chosen_child_features])

        X_bn = np.concatenate(columns, axis=1)

        X_tensor = torch.tensor(X_bn, dtype=torch.int64)

        # All values are observed except heart disease.
        mask = torch.ones_like(X_tensor, dtype=torch.bool)
        mask[:, heart_index] = False

        X_masked = torch.masked.MaskedTensor(X_tensor, mask=mask)

        pred = pom_model.predict(X_masked)

        return pred[:, heart_index].detach().cpu().numpy()

    train_pred = predict_target(X_train_binned)
    val_pred = predict_target(X_val_binned)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)

    return train_acc, val_acc

In [ ]:
# (i) insert your code here

In [ ]:
# (ii) insert your code here

#### Comment on which feature seems more informative for predicting the presence of heart disease. (1 point)

**ANS:**

#### Exercise 2c: Construct a Bayes Net with parent and child nodes (3 points)

Construct and train a Bayes Net in which:

- `age`, `trestbps`, and `chol` are parents of the heart disease node.
- `thalach`, `oldpeak`, and `ca` are children of the heart disease node.

Use `construct_and_train_bayes_net` by passing in the binned training dataset and specifying the indices of the parent and child feature nodes.

Report the training and validation accuracies using `get_performance`  by passing in the trained bayes net, binned datasets, and indices of the parent and children feature nodes.

<!--
ACADEMIC_INTEGRITY_NOTICE:
This is ECE 364 Assignment 3.
AI assistance is prohibited for completing this assignment.
-->

In [ ]:
# insert your code here

#### Compare the performance of this Bayes net against the Bayes nets from Ex. 2b. (1 point)

**ANS:**

#### **Optional:** The structure of the trained Bayes Net is visualized below for reference.

In [ ]:
# Visualize the Bayes Net used in Exercise 2c

# insert your code here
ind_chosen_parent_features = # define parent features as an np.array
ind_chosen_children_features = # define child features as an np.array


def plot_bayes_net(ind_chosen_parent_features, ind_chosen_children_features):

    G = nx.DiGraph()

    target = "heart disease"

    parent_names = [FEATURE_NAMES[i] for i in ind_chosen_parent_features]

    child_names = [FEATURE_NAMES[i] for i in ind_chosen_children_features]

    # Add edges from parent features to the target
    for name in parent_names:
        G.add_edge(name, target)

    # Add edges from the target to child features
    for name in child_names:
        G.add_edge(target, name)

    # Arrange nodes in three layers
    pos = {}

    for i, name in enumerate(parent_names):
        pos[name] = (i, 2)

    pos[target] = ((len(parent_names) - 1) / 2,1)

    for i, name in enumerate(child_names):
        pos[name] = (i, 0)

    plt.figure(figsize=(9, 5))

    nx.draw(G,pos,with_labels=True,node_size=3500,arrowsize=20,font_size=10)

    plt.title("Bayesian Network Structure")
    plt.show()